<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 2.3 — RAG with ChromaDB (Persistent Vector Store)

In this notebook, you will build a simple Retrieval-Augmented Generation (RAG) workflow using:

- **MiniLM embeddings** (local)
- **ChromaDB** (persistent vector store)
- **Ollama** (local LLM)

### Learning goals
1) Store note embeddings persistently (so you don’t rebuild every time)
2) Retrieve notes relevant to a dementia-related query
3) Use retrieved notes to extract a **dementia phenotype (Yes/No)** with evidence

> Important: We do **not** pass gold labels into retrieval or prompting. Gold labels are used only later for evaluation.

## 1. Prepare Notes Data for Embedding

We will load the cleaned notes dataset produced in Lab 6:

- `lab6_clean_notes_baseline.parquet`

This dataset is already:
- restricted to the chosen 2-year window
- deduplicated across note versions
- reduced to the most recent note per (patient, visit, note type)

In [1]:
# -----------------------------------------------------------
# 1.1. Load Cleaned Notes from Lab 6 (Parquet)
# -----------------------------------------------------------

from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

filepath = Path("data_files")

notes_df = pd.read_parquet(filepath / "lab6_clean_notes_baseline.parquet")

print("Notes loaded:", notes_df.shape)
display(notes_df.head(10))

Notes loaded: (2639, 7)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,The patient continues to experience shoulder p...
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,Sylvia was seen for follow-up regarding her sy...
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,The patient is a 69-year-old woman with a medi...
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,The patient presents with a large hiatal herni...
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This progress note documents ongoing cardiolog...
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,The care manager reached out to the patient an...
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient presented for a blood pressure che...
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\n\nThe patient is a 75-year-...
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presents for a one-year follow-up ...


## 2. Embed and Store Entire Clinical Notes in ChromaDB

In this step, we process and store each clinical note as a **complete document** in a ChromaDB vector store. This approach preserves full patient context, making it ideal for semantic search and downstream clinical reasoning.

### Step 2.1: Embed and Store Notes
1. **Embed Full Notes**
   Each note is converted into a semantic vector using a lightweight transformer model (`MiniLM`).

2. **Store in ChromaDB**
   The embedded vector and its associated metadata (e.g., patient ID, encounter number, visit date) are stored together in a persistent ChromaDB directory.

### Why This Approach?

Storing full notes is especially useful when:
- You want to retrieve the complete narrative for clinical context
- Your downstream task (e.g., summarization or decision support) depends on comprehensive input
- Each note fits within the input limit of a single LLM call

This setup supports more faithful summarization and reasoning than chunk-based approaches when notes are relatively short and self-contained.

<img src="./images/rag_full.png" alt="RAG Full" width="900">


In [2]:
# -----------------------------------------------------------
# 2.1. Embed Notes Using MiniLM and Store in Persistent ChromaDB
# -----------------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize embedding model (local)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

PERSIST_DIR = "./data_files/chroma_db_notes_epi"

# Create/load Chroma store
vectorstore = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding_model
)

# NOTE: If you re-run this cell, you may add duplicates.
# For the workshop: we will only add texts if the store is empty.
existing_count = vectorstore._collection.count()
print("Existing vectors in Chroma:", existing_count)

if existing_count == 0:
    documents = notes_df["note_txt_deid"].fillna("").tolist()
    metadata = notes_df[[
        "patient_num",
        "visit_id",
        "note_csn_id",
        "inpatient_note_type_dsc"
    ]].to_dict(orient="records")

    vectorstore.add_texts(texts=documents, metadatas=metadata)
    print(f"✅ Embedded and stored {len(documents)} notes in ChromaDB.")
else:
    print("✅ ChromaDB already populated. Skipping add_texts to avoid duplicates.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Existing vectors in Chroma: 2639
✅ ChromaDB already populated. Skipping add_texts to avoid duplicates.


## 3. Retrieving Clinical Notes with Similarity Score (RAG Retrieval with ChromaDB)

In this section, we retrieve relevant clinical notes from a ChromaDB vector store using vector similarity techniques. We explore both standard and advanced retrieval strategies to improve the relevance and diversity of retrieved context for downstream LLM generation.

### Key Retrieval Strategies

1. **Define a Clinical Query (Step 3.1)**
   - The user provides a natural language question (e.g., "Who has asthma and is taking Fluticasone and Albuterol?").

2. **Similarity Search with Scores (Step 3.2)**
   - Retrieves the top-k notes ranked by semantic similarity to the query.
   - Returns cosine-based relevance scores for each match.

3. **Score Threshold Filtering (Step 3.3)**
   - Filters out low-confidence matches using a minimum similarity score.
   - Retains only notes that meet a defined relevance threshold (e.g., ≥ 0.6).

4. **Maximal Marginal Relevance (MMR) Search (Step 3.4)**
   - Balances similarity and diversity.
   - Reduces redundancy by selecting a diverse subset of highly relevant notes.

### Why Use These Strategies?

High-quality retrieval is critical to the success of RAG workflows. These techniques help:
- Improve the contextual relevance of inputs to the LLM
- Filter out irrelevant or low-confidence documents
- Encourage diverse results to reduce bias and improve robustness

<img src="./images/rag_retrieval.png" alt="RAG Retrieval" width="900">


In [3]:
# -----------------------------------------------------------
# 3.1. Define the Query for Clinical Note Retrieval
# -----------------------------------------------------------
# This cell defines a natural language query to search the embedded clinical notes
# stored in ChromaDB using semantic similarity.

# Concept:
# The query will be *** automatically embedded *** by the vectorstore before performing
# a semantic comparison against stored note vectors.
# -----------------------------------------------------------

# Example Clinical Query:
query = "Does this patient have a documented diagnosis of dementia or Alzheimer disease?"

# Display the query for reference
display(Markdown(f"### 🔍 Query: `{query}`"))


### 🔍 Query: `Does this patient have a documented diagnosis of dementia or Alzheimer disease?`

In [4]:
# -----------------------------------------------------------
# 3.2. Perform Similarity Search with Relevance Scores
# -----------------------------------------------------------
# This cell retrieves the top-k clinical notes most semantically similar to the input query.
# Each result includes a cosine similarity score returned by LangChain's Chroma vectorstore wrapper.

# Function used:
# - vectorstore.similarity_search_with_relevance_scores(query, k=10)
#   Returns a list of (Document, score) tuples.

# Relevance Score Interpretation (heuristic only):
# - 0.90 – 1.00 → Highly relevant
# - 0.70 – 0.90 → Strong relevance
# - 0.50 – 0.70 → Moderate relevance
# - 0.30 – 0.50 → Low relevance
# - 0.00 – 0.30 → Minimal or no relevance
# -----------------------------------------------------------

# Perform the similarity search
results = vectorstore.similarity_search_with_relevance_scores(query, k=10)

# Display the results
display(Markdown("### 🔍 Retrieved Notes with Relevance Scores"))

for idx, (doc, score) in enumerate(results, 1):
    meta = doc.metadata or {}
    excerpt = (doc.page_content or "")[:900].replace("\n", " ")

    display(Markdown(
        f"---\n**Result {idx}**  \n"
        f"- **Relevance Score:** `{score:.4f}`  \n"
        f"- **patient_num:** `{meta.get('patient_num','N/A')}`  \n"
        f"- **visit_id:** `{meta.get('visit_id','N/A')}`  \n"
        f"- **note_csn_id:** `{meta.get('note_csn_id','N/A')}`  \n"
        f"- **note_type:** `{meta.get('inpatient_note_type_dsc','N/A')}`  \n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))


### 🔍 Retrieved Notes with Relevance Scores

---
**Result 1**  
- **Relevance Score:** `0.5270`  
- **patient_num:** `H116882795`  
- **visit_id:** `6371645545`  
- **note_csn_id:** `3115119523`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation....
```

---
**Result 2**  
- **Relevance Score:** `0.4868`  
- **patient_num:** `H115124574`  
- **visit_id:** `6395210747`  
- **note_csn_id:** `3382253493`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted to the inpatient unit on 12/4/2017 due to shortness of breath and congestive heart failure. Communication with the care management team was established on 12/4/2017. There is a prior diagnosis of dementia noted, for which clinical confidence is moderate. The patient remains hospitalized and is able to be contacted....
```

---
**Result 3**  
- **Relevance Score:** `0.4724`  
- **patient_num:** `H115124574`  
- **visit_id:** `6369691025`  
- **note_csn_id:** `3088256771`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted to the inpatient service on 8/24/2017. Communication with the inpatient care team was established by phone call, as preferred, and the case manager was contacted. The initial concern included anemia and consideration of heart failure. Based on previously documented information, a prior history or working diagnosis of dementia is noted. The patient is able to be contacted directly....
```

---
**Result 4**  
- **Relevance Score:** `0.4634`  
- **patient_num:** `H116088887`  
- **visit_id:** `6435733565`  
- **note_csn_id:** `3833575123`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
Progress Note  The patient, a 70-year-old right-handed female, was seen for follow-up regarding mood and cognitive difficulties. She reports variability in her days, describing both 'good days and bad days,' but notes overall mood stability. She continues to be active and enjoys attending a local day program. She visits her mother twice weekly and expresses anxiety regarding her mother's health, but maintains positive self-worth. Recent dental concerns and sleep difficulties, including trouble falling asleep and waking early, have caused some distress. Appetite remains stable, and she does not require naps.  Review of Systems: She reports mild fatigue, intermittent shortness of breath, daily headaches, memory concerns, insomnia, and subjective anxiety. All other systems were negative.  Past History: Early onset Alzheimer’s disease has been diagnosed previously, with follow-up by neurolog...
```

...

---
**Result 6**  
- **Relevance Score:** `0.4540`  
- **patient_num:** `H119641212`  
- **visit_id:** `6402407671`  
- **note_csn_id:** `4023624987`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
Visit date: 02/25/2018  This is a progress note for a 79-year-old male presenting for an annual wellness evaluation. The patient has multiple chronic health concerns, including elevated cholesterol, essential hypertension, constipation, gastroesophageal reflux disease, insomnia, low back pain, memory loss, rosacea, joint pain, vitamin D deficiency, cough, and urinary tract infection. Additional medical history includes prior cataract extractions and pain in the left great toe.  Interval Events: The patient today expresses significant concerns regarding worsening memory, with notable difficulty on cognitive testing—unable to recall three objects even immediately, although he recollects the task itself. Neurologic examination reveals trouble with expressive aphasia. He nonetheless drove himself to the clinic. There is no evidence of depression on screening, though he is anxious; anxiety ma...
```

---
**Result 7**  
- **Relevance Score:** `0.4522`  
- **patient_num:** `H115124574`  
- **visit_id:** `6412979563`  
- **note_csn_id:** `3563335851`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
A telephone call was made to the patient and spouse to arrange an initial visit. An appointment was scheduled for 3/18 at 11am, and relevant paperwork is in place. The most recent INR measurement was on 2/19, with home care services involved at the last visit. The patient is expected to visit the laboratory following the upcoming appointment. There is a prior working diagnosis of dementia that will be considered during ongoing care....
```

---
**Result 8**  
- **Relevance Score:** `0.4490`  
- **patient_num:** `H115124574`  
- **visit_id:** `6362274807`  
- **note_csn_id:** `3009596407`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
On 7/21/2017, the patient was admitted to the inpatient unit. Lip swelling was noted at the time of admission. The care management team was able to contact the patient, and communication was established. The assessment reflects mild confidence that the patient carries a prior diagnosis of dementia, though further clarification is needed. No additional interval events or findings are documented in this note....
```

---
**Result 9**  
- **Relevance Score:** `0.4410`  
- **patient_num:** `H115124574`  
- **visit_id:** `6404371571`  
- **note_csn_id:** `3475777617`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted through the emergency department for evaluation due to shortness of breath. Admission assessment and initial communication with the inpatient care management team occurred on 01/11/2018. The patient was able to be contacted at the time of evaluation. There is an established diagnosis of dementia that may influence care needs, although details regarding cognitive status were not further specified. No other clinical findings or updates were provided at this time....
```

---
**Result 10**  
- **Relevance Score:** `0.4253`  
- **patient_num:** `H122074591`  
- **visit_id:** `6419062979`  
- **note_csn_id:** `3981854181`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
Progress Note  The patient, an 87-year-old male, attended for an annual wellness evaluation accompanied by his family. The chief concern was memory loss, with a history of Alzheimer's dementia. Presently, he is reported to be happy with his living situation and is unaware of his dementia diagnosis. There are no specific complaints or symptoms noted at this visit, and he denied headache, chest pain, palpitations, shortness of breath, cough, nausea, vomiting, abdominal pain, bowel symptoms, urinary symptoms, rash, or fever. Appetite remains good and sleep has improved.  Medical history includes persistent atrial fibrillation, essential hypertension, hypothyroidism, depression, and Alzheimer's dementia. Hypertension and atrial fibrillation are well-controlled, and thyroid symptoms are absent. Depression symptoms are minimal, with PHQ2 and GAD2 indicating only occasional low mood or anxiety....
```

In [5]:
# -----------------------------------------------------------
# 3.3. Use a Retriever with a Similarity Score Threshold
# -----------------------------------------------------------
# This cell configures a retriever that only returns documents whose
# cosine similarity scores exceed a predefined threshold.

# Parameters:
# - search_type="similarity_score_threshold": Enables threshold-based filtering.
# - search_kwargs={"k": 10, "score_threshold": 0.6}
#     - k: Maximum number of documents to *evaluate* (not necessarily return)
#     - score_threshold: Minimum similarity score (0–1) required for inclusion

# Purpose:
# Improves retrieval precision by excluding weak semantic matches—
# especially important for clinical reasoning and safety-critical applications.
# -----------------------------------------------------------

# Define threshold and top-k limit
# -----------------------------------------------------------
# 3.3. Retriever with Similarity Score Threshold
# -----------------------------------------------------------

score_threshold = 0.47
top_k = 10

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": top_k, "score_threshold": score_threshold}
)

threshold_docs = retriever.invoke(query)

display(Markdown(f"### 🔎 Retrieved Notes (Score ≥ {score_threshold})"))

for idx, doc in enumerate(threshold_docs, 1):
    meta = doc.metadata or {}
    excerpt = (doc.page_content or "")[:900].replace("\n", " ")

    display(Markdown(
        f"---\n**Doc {idx}**  \n"
        f"- **patient_num:** `{meta.get('patient_num','N/A')}`  \n"
        f"- **visit_id:** `{meta.get('visit_id','N/A')}`  \n"
        f"- **note_csn_id:** `{meta.get('note_csn_id','N/A')}`  \n"
        f"- **note_type:** `{meta.get('inpatient_note_type_dsc','N/A')}`  \n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))

display(Markdown(f"**✅ Total returned:** `{len(threshold_docs)}`"))


### 🔎 Retrieved Notes (Score ≥ 0.47)

---
**Doc 1**  
- **patient_num:** `H116882795`  
- **visit_id:** `6371645545`  
- **note_csn_id:** `3115119523`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation....
```

---
**Doc 2**  
- **patient_num:** `H115124574`  
- **visit_id:** `6395210747`  
- **note_csn_id:** `3382253493`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted to the inpatient unit on 12/4/2017 due to shortness of breath and congestive heart failure. Communication with the care management team was established on 12/4/2017. There is a prior diagnosis of dementia noted, for which clinical confidence is moderate. The patient remains hospitalized and is able to be contacted....
```

---
**Doc 3**  
- **patient_num:** `H115124574`  
- **visit_id:** `6369691025`  
- **note_csn_id:** `3088256771`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted to the inpatient service on 8/24/2017. Communication with the inpatient care team was established by phone call, as preferred, and the case manager was contacted. The initial concern included anemia and consideration of heart failure. Based on previously documented information, a prior history or working diagnosis of dementia is noted. The patient is able to be contacted directly....
```

**✅ Total returned:** `3`

In [6]:
# -----------------------------------------------------------
# 3.4. Perform Maximal Marginal Relevance (MMR) Search
# -----------------------------------------------------------
# This cell retrieves clinical notes using Maximal Marginal Relevance (MMR),
# an approach that balances query relevance with result diversity.

# Parameters:
# - fetch_k = 500: Number of top-ranked candidates to evaluate before reranking
# - k = 5: Final number of diverse, relevant documents to return
# - lambda_mult:
#     - 0.0 → prioritize diversity (minimal redundancy)
#     - 1.0 → prioritize similarity to the query
#     - 0.5 → balanced trade-off between diversity and relevance

# Purpose:
# MMR reduces redundancy by penalizing near-duplicate documents, while still
# prioritizing clinical notes that are highly relevant to the query.
# This is especially useful in settings where multiple perspectives or
# treatment variations are valuable (e.g., different asthma management plans).
# -----------------------------------------------------------

# Perform MMR search
results = vectorstore.max_marginal_relevance_search(
    query=query,
    k=5,
    fetch_k=500,
    lambda_mult=0.5
)

# Display results
display(Markdown("### 📘 Retrieved Clinical Notes Using Maximal Marginal Relevance (MMR)"))

for idx, doc in enumerate(results, 1):
    patient = doc.metadata.get("patient_num", "N/A")
    date = doc.metadata.get("start_date", "N/A")
    doc_id = getattr(doc, "id", "N/A")
    excerpt = doc.page_content[:1000].replace("\n", " ")

    display(Markdown(
        f"**Document {idx}**  \n"
        f"- **Patient Num:** `{patient}`  \n"
        f"- **Document ID:** `{doc_id}`  \n\n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))

display(Markdown(f"**✅ Total MMR Results Returned:** `{len(results)}`"))


### 📘 Retrieved Clinical Notes Using Maximal Marginal Relevance (MMR)

**Document 1**  
- **Patient Num:** `H116882795`  
- **Document ID:** `b7373ef3-f711-4e20-950f-a97fb386cf6f`  

**Excerpt:**
```text
The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation....
```

**Document 2**  
- **Patient Num:** `H115648446`  
- **Document ID:** `c3b00a4d-2279-4e43-b61a-39bff61050d1`  

**Excerpt:**
```text
An 81-year-old woman returned for follow-up regarding diabetes and chronic kidney disease. She reported intermittent cough that was also noticed by her daughters. The cough is productive though she has not expectorated sputum, so is unsure of its characteristics. She does not experience shortness of breath or chest pain and is not taking any medications specifically for the cough.  Her relevant medical conditions include anemia, chronic renal failure, diabetes mellitus, hypertensive disorder, hypercholesterolemia, thyroid nodule, hyperthyroidism, and bilateral knee pain. There is a prior diagnosis of Alzheimer's disease. Family history is notable for hypertension in her mother. The patient does not smoke or use tobacco. She does drink alcohol. She is generally independent with bathing, dressing, and mobility, and rates her health as good. There were no reported changes in weight, hearing, or vision. She reports no chest pain, palpitations, abdominal pain, joint pain, or rash. She does ...
```

**Document 3**  
- **Patient Num:** `H117670807`  
- **Document ID:** `1b10ef8a-5a40-4c4d-8947-b293d49c0537`  

**Excerpt:**
```text
The patient presented for laboratory testing....
```

**Document 4**  
- **Patient Num:** `H122355386`  
- **Document ID:** `9a72ada4-de73-48fe-a92e-0916f1982d4f`  

**Excerpt:**
```text
The patient was seen for a 47-minute visit. During the session, she shared her family history, including her mother's experience balancing a business, raising four children, and having a romantic relationship outside of marriage. The patient recounted her own unsuccessful marriage and spoke about her current situation, expressing significant feelings of depression....
```

**Document 5**  
- **Patient Num:** `H115356891`  
- **Document ID:** `1180fa2a-dd99-45d9-b9b9-9f6e4218e817`  

**Excerpt:**
```text
The patient was admitted following a fall that occurred while she was in the bathtub, approximately 30 minutes before her daughter arrived. Upon admission, the patient was unable to recall the circumstances of the fall, which is notable given her prior diagnosis of dementia. Evaluation revealed a lumbar compression fracture at the L2 vertebra, confirmed by plain film imaging and MRI, with mild loss of central height. A CT scan of the spine was negative for other fractures. Laboratory findings showed a mildly elevated CPK consistent with mild rhabdomyolysis. Chronic atrial fibrillation was also present.  Imaging studies included a CT of the head, which revealed no acute intracranial bleeds but showed an old infarct in the right thalamus. MRI of the brain indicated age-related atrophy, nonspecific white matter changes, and sequelae of small vessel ischemic disease. Chest x-ray demonstrated no consolidation, and abdominal ultrasound was negative for aneurysm or gallstones. Abdominal pelvi...
```

**✅ Total MMR Results Returned:** `5`

## 4. Generating Structured Responses with an LLM (RAG Retrieval)

In this final step, we use a Large Language Model (LLM) to analyze the clinical notes retrieved in the previous section. By injecting these relevant notes into a structured prompt, we enable the LLM to generate clinically useful, structured responses.

This completes the Retrieval-Augmented Generation (RAG) pipeline, connecting search results to intelligent language generation.

### Key Steps

1. **Create a Prompt Template (Step 4.1)**
   - Defines a reusable prompt structure that includes placeholders for both the query and the retrieved clinical context.
   - Specifies a structured output format, including metadata fields such as patient ID, encounter date, and a clinical summary.

2. **Invoke the LLM with Retrieved Context (Step 4.2)**
   - Fills the prompt with retrieved notes and the clinical query.
   - Sends the prompt to a local LLM (e.g., Qwen2 or LLaMA 3 via Ollama).
   - Returns a structured, context-aware response to the clinical question.

### Why It Matters

This generation step demonstrates the power of combining semantic search with generative AI:
- Produces context-rich answers grounded in patient data
- Supports clinical summarization and decision support
- Enables zero-shot reasoning over real-world clinical notes

<img src="./images/rag_generation.png" alt="RAG Generation" width="1250">


In [7]:
# -----------------------------------------------------------
# 4.1. Prompt Template: Dementia Phenotype from Retrieved Notes
# -----------------------------------------------------------
# This chat prompt guides the LLM to generate structured clinical summaries
# from notes retrieved via similarity search.

# Placeholders:
# - {retrieved_docs}: Injects top-matching clinical notes
# - {query}: A user-defined clinical question

# Output Expectations:
# - One structured summary per patient
# - Metadata included for traceability
# - Response based only on the most recent note per patient
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a clinical phenotyping assistant. "
     "Use ONLY the provided notes. Do NOT infer or invent. "
     "Do NOT mention privacy/redaction/date shifting/removed identifiers."),
    ("human",
     "Retrieved notes:\n\n{retrieved_docs}\n\n"
     "Clinical question: {query}\n\n"
     "Return a structured answer using bullet points:\n"
     "1) Dementia Phenotype (Yes/No)\n"
     "2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')\n"
     "3) Rationale: 1–2 sentences grounded in the evidence\n"
     "4) Confidence: low/medium/high\n\n"
     "Rules for Dementia Phenotype:\n"
     "- Answer 'Yes' ONLY if dementia is explicitly documented (e.g., 'dementia', 'Alzheimer', "
     "'vascular dementia', 'Lewy body dementia') or clearly stated as a prior diagnosis.\n"
     "- If dementia is not explicitly mentioned, answer 'No' (do not infer from memory complaints alone)."
    )
])

print("✅ Prompt created and ready to use.")


✅ Prompt created and ready to use.


In [8]:
# -----------------------------------------------------------
# 4.2. Invoke LLM Using ALL Threshold-Retrieved Notes (with Traceability)
# -----------------------------------------------------------

from langchain_ollama import ChatOllama
from IPython.display import display, Markdown

MAX_CHARS_PER_NOTE = 2500

docs_for_llm = threshold_docs  # use threshold retrieval results

if len(docs_for_llm) == 0:
    display(Markdown("### No notes passed the threshold — nothing to send to the LLM."))
else:
    retrieved_blocks = []
    for rank, doc in enumerate(docs_for_llm, start=1):
        meta = doc.metadata or {}
        text = (doc.page_content or "").strip()[:MAX_CHARS_PER_NOTE]

        retrieved_blocks.append(
            f"---\n"
            f"[Retrieved Note #{rank}]\n"
            f"patient_num={meta.get('patient_num','N/A')} | visit_id={meta.get('visit_id','N/A')} | note_csn_id={meta.get('note_csn_id','N/A')}\n"
            f"note_type={meta.get('inpatient_note_type_dsc','N/A')} | create_dts_shifted={meta.get('create_dts_shifted','N/A')}\n\n"
            f"{text}\n"
        )

    retrieved_context = "\n".join(retrieved_blocks)

    display(Markdown(f"### Retrieved notes sent to LLM: {len(docs_for_llm)}"))
    display(Markdown(retrieved_context[:3000] + ("\n\n*(preview truncated)*" if len(retrieved_context) > 3000 else "")))

    final_prompt = prompt_template.format(
        retrieved_docs=retrieved_context,
        query=query
    )

    model = ChatOllama(model="llama4:scout", temperature=0.0)
    response = model.invoke(final_prompt)

    display(Markdown("### 🧠 LLM Output (Dementia Phenotype)"))
    display(Markdown(response.content))

### Retrieved notes sent to LLM: 3

---
[Retrieved Note #1]
patient_num=H116882795 | visit_id=6371645545 | note_csn_id=3115119523
note_type=Progress Notes | create_dts_shifted=N/A

The provider has reviewed the recent notes, assessments, and procedures documented by the nurse practitioner and concurs with their clinical findings. The patient's prior diagnosis of dementia was acknowledged in the reviewed documentation.

---
[Retrieved Note #2]
patient_num=H115124574 | visit_id=6395210747 | note_csn_id=3382253493
note_type=Progress Notes | create_dts_shifted=N/A

The patient was admitted to the inpatient unit on 12/4/2017 due to shortness of breath and congestive heart failure. Communication with the care management team was established on 12/4/2017. There is a prior diagnosis of dementia noted, for which clinical confidence is moderate. The patient remains hospitalized and is able to be contacted.

---
[Retrieved Note #3]
patient_num=H115124574 | visit_id=6369691025 | note_csn_id=3088256771
note_type=Progress Notes | create_dts_shifted=N/A

The patient was admitted to the inpatient service on 8/24/2017. Communication with the inpatient care team was established by phone call, as preferred, and the case manager was contacted. The initial concern included anemia and consideration of heart failure. Based on previously documented information, a prior history or working diagnosis of dementia is noted. The patient is able to be contacted directly.


### 🧠 LLM Output (Dementia Phenotype)

Here is the structured answer:

* 1) Dementia Phenotype: Yes
* 2) Evidence: 
  - "prior diagnosis of dementia" (Note #1)
  - "a prior diagnosis of dementia" (Note #2)
  - "a prior history or working diagnosis of dementia" (Note #3)
* 3) Rationale: The patient's medical notes explicitly mention a prior diagnosis of dementia, which supports the presence of this condition. The repeated mention of dementia across different notes adds to the confidence in this assessment.
* 4) Confidence: high

In [9]:
# -----------------------------------------------------------
# 4.3 (Bonus). Patient-Level Dementia Phenotyping from Notes
# -----------------------------------------------------------
# Features:
# - Processes patients sequentially
# - Early stop when dementia detected
# - Progress logging
# - Optional patient limit for faster runs
# -----------------------------------------------------------

import pandas as pd
from pathlib import Path
from langchain_ollama import ChatOllama

# Load evaluation labels (for later merge)
eval_df = pd.read_csv(Path("data_files") / "lab6_structured_dementia_eval.csv")

# LLM model
model = ChatOllama(model="qwen2", temperature=0.0)

# -----------------------------------------------------------
# OPTIONAL: limit number of patients (set to None to run all)
# -----------------------------------------------------------

PATIENT_LIMIT = 5   # change to None to process all patients

unique_patients = notes_df["patient_num"].unique().tolist()

if PATIENT_LIMIT:
    unique_patients = unique_patients[:PATIENT_LIMIT]

total_patients = len(unique_patients)

print(f"Patients to process: {total_patients}\n")

results = []

for idx, patient_num in enumerate(unique_patients, start=1):

    print(f"Processing patient {idx} of {total_patients} -> {patient_num}")

    patient_notes = notes_df[notes_df["patient_num"] == patient_num]

    dementia_found = False
    notes_checked = 0
    first_yes_note_csn = None
    first_yes_visit_id = None
    evidence_preview = None

    for _, row in patient_notes.iterrows():

        notes_checked += 1

        note_text = (row["note_txt_deid"] or "").strip()
        if len(note_text) == 0:
            continue

        retrieved_docs = (
            f"patient_num={row.get('patient_num','N/A')} | visit_id={row.get('visit_id','N/A')} | "
            f"note_csn_id={row.get('note_csn_id','N/A')}\n"
            f"note_type={row.get('inpatient_note_type_dsc','N/A')} | "
            f"create_dts_shifted={row.get('create_dts_shifted','N/A')}\n\n"
            f"{note_text}"
        )

        query = "Does the patient have a documented diagnosis of dementia or Alzheimer disease?"

        final_prompt = prompt_template.format(
            retrieved_docs=retrieved_docs,
            query=query
        )

        response = model.invoke(final_prompt).content

        # simple detection
        if "Dementia Phenotype" in response and "Yes" in response.split("Dementia Phenotype", 1)[1][:60]:

            dementia_found = True
            first_yes_note_csn = row.get("note_csn_id")
            first_yes_visit_id = row.get("visit_id")
            evidence_preview = response[:400]

            break

    print(f"   dementia: {'YES' if dementia_found else 'NO'}")
    print(f"   notes evaluated: {notes_checked}\n")

    results.append({
        "patient_num": patient_num,
        "rag_dementia": dementia_found,
        "notes_checked": notes_checked,
        "first_yes_note_csn_id": first_yes_note_csn,
        "first_yes_visit_id": first_yes_visit_id,
        "rag_output_preview": evidence_preview
    })

rag_df = pd.DataFrame(results)

print("RAG results shape:", rag_df.shape)
display(rag_df.head(10))

Patients to process: 5

Processing patient 1 of 5 -> H120513333
   dementia: NO
   notes evaluated: 31

Processing patient 2 of 5 -> H111336931
   dementia: NO
   notes evaluated: 17

Processing patient 3 of 5 -> H120897999
   dementia: NO
   notes evaluated: 36

Processing patient 4 of 5 -> H122074591
   dementia: YES
   notes evaluated: 2

Processing patient 5 of 5 -> H122355386
   dementia: NO
   notes evaluated: 176

RAG results shape: (5, 6)


,patient_num,rag_dementia,notes_checked,first_yes_note_csn_id,first_yes_visit_id,rag_output_preview
0,H120513333,False,31,NaN,NaN,NaN
1,H111336931,False,17,NaN,NaN,NaN
2,H120897999,False,36,NaN,NaN,NaN
3,H122074591,True,2,2.507761e+09,6.297519e+09,"- Dementia Phenotype: Yes\n- Evidence: ""Alzhei..."
4,H122355386,False,176,NaN,NaN,NaN


In [10]:
# -----------------------------------------------------------
# 4.4 (Bonus). Merge RAG phenotype with structured + gold labels
# -----------------------------------------------------------

merged = eval_df.merge(rag_df, on="patient_num", how="left")

print("Merged shape:", merged.shape)
display(merged.head(10))

# Simple cross-tab vs gold
conf = pd.crosstab(
    merged["rag_dementia"].astype(bool),
    merged["gold_dementia"].astype(bool),
    rownames=["RAG Dementia"],
    colnames=["Gold Dementia"]
)
display(conf)

Merged shape: (97, 11)


,patient_num,dx_dementia_flag,med_dementia_flag,baseline_dementia,gold_diagnosis,gold_dementia,rag_dementia,notes_checked,first_yes_note_csn_id,first_yes_visit_id,rag_output_preview
0,H122355386,False,False,False,MCI vs. dementia,False,False,176.0,NaN,NaN,NaN
1,H116896574,False,False,False,Dementia,True,NaN,NaN,NaN,NaN,NaN
2,H112639340,False,False,False,No MCI or dementia diagnosis (normal),False,NaN,NaN,NaN,NaN,NaN
3,H117728724,False,False,False,No MCI or dementia diagnosis (normal),False,NaN,NaN,NaN,NaN,NaN
4,H116273351,False,False,False,No MCI or dementia diagnosis (normal),False,NaN,NaN,NaN,NaN,NaN
5,H113296266,False,False,False,No MCI or dementia diagnosis (normal),False,NaN,NaN,NaN,NaN,NaN
6,H112117815,False,True,True,Dementia,True,NaN,NaN,NaN,NaN,NaN
7,H115421676,False,False,False,No MCI or dementia diagnosis (normal),False,NaN,NaN,NaN,NaN,NaN
8,H115051235,False,False,False,No MCI or dementia diagnosis (normal),False,NaN,NaN,NaN,NaN,NaN
9,H111969872,False,False,False,Dementia,True,NaN,NaN,NaN,NaN,NaN


Gold Dementia,False,True
RAG Dementia,,
False,4,0
True,68,25
